In [2]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))


from scripts.text_matching import normalise_text

In [3]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
still_unmatched_file = Path(project_root / "data/people/still_unmatched.json")

In [4]:
with open(still_unmatched_file , "r") as f:
   still_unmatched = json.load(f)
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)

In [5]:

variants_dict = {}
for variant in variants:
    variants_dict[normalise_text(variant["variant_normalised"])] = variant["person_id"]

rprint(dict(list(variants_dict.items())[:5]))

people_dict = {}

for person in people:
    last = person["family_name"]
    first = person["given_names"]

    if last is None or first is None:
        continue

    last_norm = normalise_text(last)
    first_norm = normalise_text(first)

    key = (last_norm, first_norm)
    people_dict[key] = person["person_id"]

# rprint(dict(list(people_dict.items())[:5]))

single_dict = {}

for person in people:
    single = person["single_name"]
    if not single:
        continue
    else:
        single_norm = normalise_text(single)

    single_dict[single_norm] = person["person_id"]


{'abel, othenio': 1, 'abel, otto': 2, 'otto abel u. wilhelm wattenbach': 2, 'kurt absolon': 3, 'abu temmam': 4}

In [6]:
SEPARATORS = re.compile(r"\s*;\s*|\s+/\s+|\s+und\s+")
ETC_SUFFIX = re.compile(r"\s+u\.a\.\s*$")

matched_via_variants = {}
still_unmatched_v2 = {}

for name, entries in still_unmatched.items():
    # Strip "u.a." trailing tag, then split on multi-person separators
    cleaned = ETC_SUFFIX.sub("", name).strip()
    parts = [p.strip() for p in SEPARATORS.split(cleaned) if p.strip()]

    person_ids = []
    failed_parts = []

    for part in parts:
        # Try 1: full-string variant lookup
        person_id = variants_dict.get(part)

        # Try 2: fall back to your existing first/last/single logic
        if not person_id:
            if ", " in part:
                last, first = part.split(", ", 1)
                person_id = people_dict.get((last.strip(), first.strip()))
            else:
                split = part.rsplit(" ", 1)
                if len(split) == 2:
                    first, last = split
                    person_id = people_dict.get((last, first))
                else:
                    person_id = single_dict.get(part)

        if person_id:
            person_ids.append(person_id)
        else:
            failed_parts.append(part)

    if person_ids and not failed_parts:
        matched_via_variants[name] = {"person_ids": person_ids, "entries": entries}
    else:
        still_unmatched_v2[name] = {
            "person_ids": person_ids,
            "failed_parts": failed_parts,
            "entries": entries,
        }

print(f"Newly matched: {len(matched_via_variants)}")
print(f"Still unmatched: {len(still_unmatched_v2)}")

rprint(matched_via_variants)
with open("multi_person_entries_split.json", "w") as f:
    json.dump(matched_via_variants, f, ensure_ascii=False, indent=2)
# with open("still_unmatched_v2_file.json", "w") as f:
#     json.dump(still_unmatched_v2, f, ensure_ascii=False, indent=2)

Newly matched: 34
Still unmatched: 1575


{
    'wilhelm südel und friedrich burschell': {
        'person_ids': [6885, 937],
        'entries': [
            {
                'display_name': 'Wilhelm Südel und Friedrich Burschell',
                'composite_id': 'fremdsprachige_720_29_38',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'gegenschatz, ernst; gigon, olof': {
        'person_ids': [2096, 2174],
        'entries': [
            {
                'display_name': 'Gegenschatz, Ernst; Gigon, Olof',
                'composite_id': 'antike_29_2_9',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'gegenschatz, ernst / gigon, olof': {
        'person_ids': [2096, 2174],
        'entries': [
            {
                'display_name': 'Gegenschatz, Ernst / Gigon, Olof',
                'composite_id': 'antike_30_2_9',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'urban, peter; ziegler, rosemarie; artmann, h. c.; celan, paul; mayröcker, friederike u.a.': {
        'person_ids': [7149, 7770, 171, 1029, 4511],
        'entries': [
            {
                'display_name': 'Urban, Peter; Ziegler, Rosemarie; Artmann, H. C.; Celan, Paul; Mayröcker, 
Friederike u.a.',
                'composite_id': 'russland_79_4_17',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'therre, hans; schmidt, rainer g.': {
        'person_ids': [6953, 6148],
        'entries': [
            {
                'display_name': 'Therre, Hans; Schmidt, Rainer G.',
                'composite_id': 'briefe_308_13_15',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'drohla, gisela; eliasberg, alexander; luther, arthur; röhl, hermann': {
        'person_ids': [1471, 1610, 4269, 5721],
        'entries': [
            {
                'display_name': 'Drohla, Gisela; Eliasberg, Alexander; Luther, Arthur; Röhl, Hermann',
                'composite_id': 'russland_352_15_17',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'karla hielscher u.a.': {
        'person_ids': [2866],
        'entries': [
            {
                'display_name': 'Karla Hielscher u.a.',
                'composite_id': 'russland_355_15_17',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'hans halm und richard hoffmann': {
        'person_ids': [2554, 2954],
        'entries': [
            {
                'display_name': 'Hans Halm und Richard Hoffmann',
                'composite_id': 'russland_362_15_17',
                'sort_order': 1,
                'is_author': False,
                'is_editor': False,
                'is_contributor': False,
                'is_translator': True
            }
        ]
    },
    'markus kessel und myriam alfano': {
        'person_ids': [3436, 64],
        'entries': [
            {
                'display_name': 'Markus Kessel und Myriam Alfano',
                'composite_id': 'buch_120_